# GOES-18 ABI True Color — Full Disk, then a domain

True Color is close to what the eye would see, so it needs daylight. ABI has no green band, so green is synthesised from the blue, red and vegetation channels.

Same order as the other notebooks: load the files, plot the **complete Full
Disk**, then choose a **domain** and plot that.


## Environment

Everything runs in the environment described by
[`environment.yml`](../environment.yml) at the repository root:

```bash
conda env create -f environment.yml
conda activate goes-viirs
python -m jupyter lab
```


## Setup

In [ ]:
import logging
import sys
import warnings
from pathlib import Path

from IPython.display import Image, display
from satpy import Scene
from satpy.utils import PerformanceWarning

logging.getLogger("pyspectral.rsr_reader").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=PerformanceWarning)
warnings.filterwarnings("ignore", message="Mean of empty slice", category=RuntimeWarning)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from examples.domains import DOMAINS
from examples.goes18_coverage_data import SCAN_LABELS, download_coverage
from examples.render_satellite import (
    crop_and_resample_scene,
    resample_to_max_size,
    save_band_map,
    save_dataset_with_lonlat_grid,
    save_map,
)


## 1. Get the data

True Color needs these channels from one scan.

In [ ]:
COVERAGE = "full_disk"
CHANNELS = ("C01", "C02", "C03",)

# Where the NetCDF files live. download_coverage() fetches anything missing
# from the public NOAA bucket and reuses whatever is already on disk.
DATA_DIR = REPO_ROOT / "data" / "goes18-20231003-1900"
files = download_coverage(DATA_DIR, COVERAGE, CHANNELS)

print(f"folder: {DATA_DIR / COVERAGE}")
print(f"{len(files)} file(s):")
for name in files:
    print("  ", Path(name).name)


## 2. Build the composite and plot the complete Full Disk

In [ ]:
COMPOSITE = "true_color"
scene = Scene(reader="abi_l1b", filenames=files)
scene.load([COMPOSITE], generate=True)

full = resample_to_max_size(scene, COMPOSITE, max_size=1400)
out_full = REPO_ROOT / "output" / "full_disk_true_color.png"
save_dataset_with_lonlat_grid(
    full, COMPOSITE, out_full,
    title=f"GOES-18 ABI True Color — Full Disk — {SCAN_LABELS['full_disk']}",
    dpi=100,
)
display(Image(filename=str(out_full)))


## 3. Choose a domain

In [ ]:
DOMAIN_NAME = "shishaldin_big"
DOMAIN = DOMAINS[DOMAIN_NAME]
RESOLUTION = 0.02

print(f"{DOMAIN_NAME}: {DOMAIN}")


## 4. Plot that domain

In [ ]:
cropped = crop_and_resample_scene(scene, domain=DOMAIN, resolution=RESOLUTION)
try:
    cropped[COMPOSITE]
except KeyError:
    cropped.load([COMPOSITE], generate=True)

out_domain = REPO_ROOT / "output" / "domain_true_color.png"
save_map(cropped, COMPOSITE, out_domain, title="G18 - True Color")
display(Image(filename=str(out_domain)))


## Reading the colours

Ocean is dark blue, land green to brown, cloud and snow white. The brown streak south of Shishaldin is the volcanic ash plume.

See [docs/RGB.md](../docs/RGB.md) for the full recipes.
